## Gradient Descent and Streaming Chat

**Week 08 — Revision Notebook**

এই notebook-এ তিনটা জিনিস আছে: **Math**, **Theory** আর আমার নিজের **Code**।
কিছু ভুলে গেলে এখানে এসে পড়ে নিলেই হবে।
প্রতিটা topic-এ একই নিয়ম: মানে → ছোট উদাহরণ → AI-তে কেন লাগে।

### সূচিপত্র (Table of Contents)

**১. MATH (Calculus II)**
- [Loss Function](#s1)
- [Slope বলে কোন দিকে যেতে হবে](#s2)
- [Gradient Descent](#s3)
- [Learning Rate](#s4)
- [Gradient (দুই variable)](#s5)
- [Local ও Global Minimum](#s6)
- [Softmax](#s7)
- [Cross Entropy Loss](#s8)

**২. THEORY (Streaming ও Chat)**
- [Streaming](#s9)
- [Generator (yield বনাম return)](#s10)
- [SSE (Server-Sent Events)](#s11)
- [Conversation History](#s12)
- [Context Window ও খরচ](#s13)
- [Chat Persistence](#s14)
- [Session Management](#s15)
- [Error Handling](#s16)
- [CORS](#s17)
- [Frontend Streaming UI](#s18)

**৩. CODE (আমার নিজের লেখা)**
- [Setup: openai install](#s19)
- [Setup: client](#s20)
- [Loss table: f(x) = (x−3)²](#s21)
- [Slope: 2·(x−3)](#s22)
- [Gradient Descent loop: x=5, lr=0.001](#s23)
- [Gradient (দুই variable): x=4, y=2, lr=0.1](#s24)
- [Local ও Global: f(x) = x⁴ − 3x² + x এর table](#s25)
- [Local ও Global: দুই জায়গা থেকে শুরু](#s26)
- [Local ও Global: slope(−2) যাচাই](#s27)
- [Softmax: scores [5.0, 1.0, 0.1]](#s28)
- [Cross Entropy: দুই softmax-এর loss](#s29)
- [Cross Entropy: p বদলালে loss](#s30)
- [Streaming: stream_text() with yield](#s31)
- [Generator: count_to()](#s32)
- [Generator: yield-এর মাঝে থামা](#s33)
- [SSE: to_sse()](#s34)
- [ask(): history ছাড়া](#s35)
- [chat(): history সহ](#s36)
- [history-র দৈর্ঘ্য](#s37)
- [history-র ভেতরে কী আছে](#s38)
- [history[-N:]: শেষের N টা message](#s39)
- [পরীক্ষা: নাম মনে থাকে কি?](#s40)
- [save_history() আর load_history()](#s41)
- [History আবার load করা](#s42)
- [sessions: user-ভেদে আলাদা history](#s43)
- [safe_chat(): প্রথম version](#s44)
- [safe_chat(): retry সহ](#s45)

**শেষে**
- [৪. আমার নোট](#s46)

<a id="p1"></a>

## ১. Math: Calculus II

Gradient descent মানে slope ধরে ধরে নিচের দিকে নামা। Model এভাবেই শেখে।

<a id="s1"></a>

### Loss Function

**মানে:** Loss হলো model কতটা ভুল করছে, তার সংখ্যা। যত ছোট, তত ভালো।

**উদাহরণ:** f(x) = (x−3)²। x=0 হলে 9, x=2 হলে 1, x=3 হলে 0। তাই minimum হলো x=3।

**AI-তে কেন লাগে:** model training-এর কাজই হলো এই loss ছোট করা।

<a id="s2"></a>

### Slope বলে কোন দিকে যেতে হবে

**মানে:** Slope বলে দেয় কোন দিকে গেলে loss বাড়ে। আমরা উল্টো দিকে যাই। এখানে slope = 2·(x−3)।

**উদাহরণ:**
- x=5 হলে slope = 4 (positive) → x **কমাও**।
- x=0 হলে slope = −6 (negative) → x **বাড়াও**।
- x=3 হলে slope = 0 → থামো, minimum পেয়ে গেছি।

**AI-তে কেন লাগে:** weight বাড়াতে হবে না কমাতে হবে, slope-ই সেটা বলে দেয়।

<a id="s3"></a>

### Gradient Descent

**মানে:** Slope দেখে ছোট ছোট পা ফেলে minimum-এ পৌঁছানো।

```
x = x − lr × slope
```

**উদাহরণ:** x=5 থেকে শুরু, lr=0.1, মোট 10 ধাপ।

| ধাপ | slope | step (lr × slope) | নতুন x |
|---|---|---|---|
| 1 | 4 | **0.4** | 4.6 |
| 2 | 3.2 | **0.32** | 4.28 |
| 3 | 2.56 | **0.256** | 4.024 |
| … | … | … | … |
| 10 | 0.537 | 0.0537 | 3.21 |

Minimum-এর কাছে slope ছোট হয়, তাই step-ও ছোট হয়। এজন্য শেষে আস্তে আস্তে থামে।

**AI-তে কেন লাগে:** neural network এভাবেই বারবার weight বদলে শেখে।

<a id="s4"></a>

### Learning Rate

**মানে:** lr হলো পায়ের মাপ, অর্থাৎ এক ধাপে কতটা এগোবো।

**উদাহরণ:** x=5 থেকে 10 ধাপ পর (minimum হলো 3):

| lr | 10 ধাপ পর x | কী হলো |
|---|---|---|
| 0.001 | 4.96 | খুব ধীর। এখনো অনেক দূরে। |
| 0.1 | 3.21 | ভালো। 3-এর দিকে যাচ্ছে। |
| 1.1 | 15.38 | ভুল! লাফিয়ে দূরে চলে গেল (diverge)। |

lr=1.1 হলে x এভাবে চলে: 5 → 0.6 → 5.88 → −0.46 → 7.15। প্রতিবার 3 থেকে দূরত্ব বাড়ে।

**AI-তে কেন লাগে:** lr ঠিক না হলে model হয় শেখে না, নয়তো loss উল্টে বাড়তে থাকে।

<a id="s5"></a>

### Gradient (দুই variable)

**মানে:** Variable দুটো হলে প্রতিটার আলাদা slope থাকে। এই slope-গুলোর list-ই gradient।

**উদাহরণ:** f(x, y) = x² + y², x=4, y=2।
- x-এর slope = 2x = 8, y-এর slope = 2y = 4 → gradient = **[8, 4]**
- lr=0.1 দিয়ে এক ধাপ: x = 4 − 0.1×8 = 3.2, y = 2 − 0.1×4 = 1.6 → **(3.2, 1.6)**

**AI-তে কেন লাগে:** model-এ হাজার হাজার weight থাকে। Gradient সবার slope একসাথে দেয়।

<a id="s6"></a>

### Local ও Global Minimum

**মানে:** Local minimum হলো ছোট গর্ত। Global minimum হলো সবচেয়ে গভীর গর্ত। Gradient descent শুধু নিচের দিকে হাঁটে, তাই ছোট গর্তে আটকে যেতে পারে।

**উদাহরণ:** f(x) = x⁴ − 3x² + x
- x=2 থেকে শুরু → শেষে প্রায় **1.13** (local, f ≈ −1.07)
- x=−2 থেকে শুরু → শেষে প্রায় **−1.30** (global, f ≈ −3.51)

শুরুর জায়গাই ঠিক করে কোথায় পৌঁছাবো।

**AI-তে কেন লাগে:** training-এ শুরুর weight আলাদা হলে ফলও আলাদা হতে পারে।

<a id="s7"></a>

### Softmax

**মানে:** Score-কে probability বানায়। সবগুলোর যোগফল 1 হয়।

**উদাহরণ:** scores = [2.0, 1.0, 0.1]
- e^score = [7.39, 2.72, 1.11], যোগফল = 11.21
- ভাগ করলে = **[0.659, 0.242, 0.099]**, যোগফল = 1

Score-এর ফাঁক বাড়ালে model বেশি confident হয়: [5.0, 1.0, 0.1] → [0.975, 0.018, 0.007]।

**AI-তে কেন লাগে:** LLM পরের শব্দের জন্য প্রতিটা শব্দের probability এভাবে বের করে।

<a id="s8"></a>

### Cross Entropy Loss

**মানে:** সঠিক উত্তরের probability p হলে loss = −log(p)। p ছোট হলে loss বড়।

**উদাহরণ:**

| p (সঠিক উত্তরের) | loss |
|---|---|
| 0.9 | 0.11 |
| 0.5 | 0.69 |
| 0.1 | 2.30 |
| 0.01 | 4.61 |

**AI-তে কেন লাগে:** LLM training-এ এই loss-ই কমানো হয়। ঠিক উত্তরে যত নিশ্চিত, loss তত কম।

<a id="p2"></a>

## ২. Theory: Streaming ও Chat

Streaming মানে উত্তর টুকরো টুকরো করে পাওয়া, আর Chat মানে আগের কথা সহ কথোপকথন চালানো।

`stream_server.py` (FastAPI server) আর `chat.html` (browser UI) আলাদা file। এই notebook-এর বাইরে `Phase_02` folder-এ আছে।

<a id="s9"></a>

### Streaming

**মানে:** পুরো উত্তরের জন্য না থেমে টুকরো টুকরো করে পাঠানো। `stream=True` দিলে প্রতিটা `chunk.choices[0].delta.content`-এ একটু করে লেখা আসে।

**উদাহরণ:** উত্তর বানাতে 10 সেকেন্ড লাগে। Streaming ছাড়া 10 সেকেন্ড ফাঁকা পর্দা। Streaming-এ আধ সেকেন্ডেই প্রথম শব্দ আসে। মোট সময় একই, কিন্তু দ্রুত মনে হয়।
`delta.content` কখনো `None` হয়, তাই `if piece:` লাগে।

**AI-তে কেন লাগে:** user অপেক্ষা না করে সাথে সাথে পড়া শুরু করতে পারে।

<a id="s10"></a>

### Generator (yield বনাম return)

**মানে:** `return` একবারে সব দিয়ে শেষ করে। `yield` একটা একটা করে দেয়, আর মাঝে থেমে থাকে।

**উদাহরণ:** Restaurant-এর waiter।
- `return` = সব খাবার রান্না শেষ হলে একসাথে আনা।
- `yield` = একটা খাবার হলেই একটা এনে দেওয়া।

`count_to(3)` একবারে 1, 2, 3 দেয় না। `for` চাইলে একটা একটা করে দেয়।

**AI-তে কেন লাগে:** `stream_text()` প্রতিটা টুকরো `yield` করে, তাই আমরা সাথে সাথে পাই।

<a id="s11"></a>

### SSE (Server-Sent Events)

**মানে:** Server থেকে browser-এ টুকরো টুকরো data পাঠানোর নিয়ম। প্রতিটা message দেখতে এমন:

```
data: ...\n\n
```

দুটো newline কেন? প্রথম `\n` লাইন শেষ করে। দ্বিতীয় `\n` মানে "এই message শেষ"। দুটো না পেলে browser message ধরে না।

**উদাহরণ:** `to_sse("আ")` → `'data: আ\n\n'`

**AI-তে কেন লাগে:** `chat.html` এই format-ই পড়ে। Server হলো `stream_server.py`।

<a id="s12"></a>

### Conversation History

**মানে:** LLM-এর কোনো memory নেই। প্রতিবার পুরো কথোপকথনের list পাঠাতে হয়।

**উদাহরণ:** শুধু "আমার নাম কী?" পাঠালে উত্তর: "জানি না"। আগের message সহ পাঠালে উত্তর: "রফিক"।
দুই প্রশ্নের পর `history`-তে 4টা message থাকে (user, assistant, user, assistant)।

**AI-তে কেন লাগে:** Chatbot "মনে রাখে" মনে হয়, কিন্তু আসলে প্রতিবার পুরোনো কথা আবার পাঠানো হয়।

<a id="s13"></a>

### Context Window ও খরচ

**মানে:** Model একবারে সীমিত token পড়তে পারে। বেশি token মানে বেশি খরচ। তাই `history[-N:]` দিয়ে শুধু শেষের N টা message পাঠাই।

**উদাহরণ:** 5টা প্রশ্নের শেষে `history`-তে 9টা message। `N=4` হলে প্রথম দুটো (যেখানে নাম বলা ছিল) বাদ পড়ে। তখন "আমার নাম কী?" এর উত্তর model দিতে পারে না।
আমার code-এ `MAX_MESSAGES = 10`, তাই সব message যায় আর নাম মনে থাকে। ভুলে যাওয়া দেখতে সংখ্যাটা ছোট করে (যেমন 4) চালাও।

**AI-তে কেন লাগে:** খরচ আর সময় কমে, আর context-এর সীমা পার হয় না। তবে পুরোনো কথা হারায়।

<a id="s14"></a>

### Chat Persistence

**মানে:** `history` শুধু memory-তে থাকলে program বন্ধ হলেই হারায়। তাই file-এ save করি, পরে load করি।

**উদাহরণ:** `json.dump(history, f, ensure_ascii=False, indent=2)`
- `ensure_ascii=False` না দিলে বাংলা `\u0986...` হয়ে যায়। দিলে বাংলাই লেখা থাকে।
- file খোলার সময় `encoding="utf-8"` দিতে হয়।

`chat.json`-এ ব্যক্তিগত কথা থাকে। তাই এটা `.gitignore`-এ রাখতে হবে।

**AI-তে কেন লাগে:** user আবার এলে আগের কথোপকথন ফিরিয়ে আনা যায়।

<a id="s15"></a>

### Session Management

**মানে:** অনেক user হলে প্রত্যেকের আলাদা history লাগে। `sessions` নামের dict-এ `session_id` দিয়ে আলাদা রাখি।
`sessions.setdefault(session_id, [])` মানে: আছে তো সেটাই দাও, না থাকলে নতুন খালি list বানাও।

**উদাহরণ:** `chat("rafiq", "আমার নাম রফিক।")` বললাম। তারপর `chat("karim", "আমার নাম কী?")` জিজ্ঞেস করলাম। karim-এর history আলাদা, তাই model নাম জানে না।

**AI-তে কেন লাগে:** একটা server একসাথে অনেক user সামলায়, কিন্তু কথা মিশে যায় না।

<a id="s16"></a>

### Error Handling

**মানে:** API-র সব error এক রকম নয়। কোনোটায় আবার চেষ্টা করা যায়, কোনোটায় যায় না।

| Code | Exception | কী করব |
|---|---|---|
| 401 | `AuthenticationError` | key ভুল। **Retry নয়।** সাথে সাথে থামো। |
| 429 | `RateLimitError` | বেশি request। একটু **অপেক্ষা করে retry** করো। |

**উদাহরণ:** `safe_chat(..., retries=2)` মোট 3 বার চেষ্টা করে। 429 এলে `time.sleep(2)` করে আবার চেষ্টা। 401 এলে সাথে সাথে ভদ্র message ফেরত দেয়।

**AI-তে কেন লাগে:** API ফেল করলেও app ভাঙে না, আর user একটা ভদ্র message পায়।

<a id="s17"></a>

### CORS

**মানে:** Browser নিজের নিরাপত্তার জন্য এক জায়গার page-কে অন্য address-এর server-এ request পাঠাতে দেয় না। Server অনুমতি দিলে তবেই দেয়।

**উদাহরণ:** `chat.html` একটা file, আর server চলছে `127.0.0.1:8000`-এ। Server অনুমতি না দিলে browser request আটকে দেয়।
`allow_origins` বলে কোন কোন site আসতে পারবে। `["*"]` মানে সবাই। এটা শুধু শেখার জন্য। আসল app-এ নির্দিষ্ট site-এর নাম দিতে হয়।

**AI-তে কেন লাগে:** frontend আর API server আলাদা জায়গায় থাকলে CORS ছাড়া chat চলেই না।

<a id="s18"></a>

### Frontend Streaming UI

**মানে:** Browser-এর `EventSource` server-এর stream শোনে। প্রতিটা `data:` এলে `onmessage` চলে, আর টুকরোটা পর্দায় যোগ হয়।

**উদাহরণ:** `chat.html`-এ
- পাঠাও চাপলে **loading**: "ভাবছি..." দেখায়, button বন্ধ থাকে।
- প্রথম টুকরো এলে "ভাবছি..." মুছে যায়।
- `onerror` হলে `es.close()` করে button আবার চালু করে। কোনো টুকরোই না এলে **error** message দেখায়।

**AI-তে কেন লাগে:** ChatGPT-র মতো অক্ষর একটা একটা করে ভেসে ওঠে, আর কিছু ভুল হলে user বুঝতে পারে।

<a id="p3"></a>

## ৩. Code: আমার নিজের লেখা

এখানের সব code আমার নিজের লেখা। ক্রম: setup → math → streaming → chat history → session → error handling।

<a id="s19"></a>

### Setup: openai install

প্রথমবার `openai` package লাগে।

In [1]:
%pip install openai

Note: you may need to restart the kernel to use updated packages.


<a id="s20"></a>

### Setup: client

Groq-এর API key `GROQ_API_KEY` environment variable থেকে পড়া হয়। Code-এ key লেখা নেই।

In [2]:
from openai import OpenAI
import os

client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key=os.environ["GROQ_API_KEY"],
)

<a id="s21"></a>

### Loss table: f(x) = (x−3)²

x বদলালে loss কত হয়, তার তালিকা। Minimum x=3 এ।

In [3]:
def f(x):
    return (x - 3) ** 2

for x in [0, 1, 2, 3, 4, 5]:
    print(x, f(x))

0 9
1 4
2 1
3 0
4 1
5 4


<a id="s22"></a>

### Slope: 2·(x−3)

কোন x-এ slope কত। x=3 এ slope 0।

In [4]:
def slope(x):
    return 2 * (x - 3)

for x in [0, 1, 3, 5]:
    print(x, slope(x))

0 -6
1 -4
3 0
5 4


<a id="s23"></a>

### Gradient Descent loop: x=5, lr=0.001

এখানে lr খুব ছোট, তাই 10 ধাপে x মোটে 5 থেকে 4.96 হয়েছে। (loop-এর ভেতরে `slope` নামটা সংখ্যা হয়ে যায়, আগের `slope` function ঢাকা পড়ে।)

In [5]:
x = 5
lr = 0.001

for i in range(10):

    slope = 2 * (x -3)
    x = x - lr * slope

    print(i+1, x)

1 4.996
2 4.992008
3 4.988023984
4 4.984047936032
5 4.980079840159936
6 4.976119680479616
7 4.972167441118657
8 4.96822310623642
9 4.964286660023947
10 4.960358086703899


<a id="s24"></a>

### Gradient (দুই variable): x=4, y=2, lr=0.1

দুটো variable একসাথে বদলায়। প্রথম ধাপে (3.2, 1.6)।

In [6]:
x = 4
y = 2
lr = 0.1

for i in range(10):
    grad_x = 2 * x
    grad_y = 2 * y

    x = x - lr * grad_x
    y = y - lr * grad_y

    print(i+1,x, y)

1 3.2 1.6
2 2.56 1.28
3 2.048 1.024
4 1.6384 0.8192
5 1.31072 0.65536
6 1.0485760000000002 0.5242880000000001
7 0.8388608000000002 0.4194304000000001
8 0.6710886400000001 0.33554432000000006
9 0.5368709120000001 0.26843545600000007
10 0.4294967296000001 0.21474836480000006


<a id="s25"></a>

### Local ও Global: f(x) = x⁴ − 3x² + x এর table

In [7]:
def f(x):
    return x ** 4 - 3 * x ** 2 + x

for x in [-2, -1.5, -1, 0, 1, 1.5, 2]:
    print(x, f(x))

-2 2
-1.5 -3.1875
-1 -3
0 0
1 -1
1.5 -0.1875
2 6


<a id="s26"></a>

### Local ও Global: দুই জায়গা থেকে শুরু

x=2 থেকে শুরু করলে প্রায় 1.13, x=−2 থেকে শুরু করলে প্রায় −1.30।

In [8]:
def slope(x):
    return 4 * x ** 3 - 6 * x + 1

for start in [2, -2]:
    x = start
    for i in range(1000):
        x = x - 0.01 * slope(x)
    print(start, x)

2 1.130901122629987
-2 -1.300839565941578


<a id="s27"></a>

### Local ও Global: slope(−2) যাচাই

In [9]:
def slope(x):
    return 4 * x ** 3 - 6 * x + 1

print(slope(-2))

-19


<a id="s28"></a>

### Softmax: scores [5.0, 1.0, 0.1]

ফাঁক বেশি, তাই প্রথমটার probability প্রায় 0.975।

In [10]:
import math

scores = [5.0, 1.0, 0.1]

exps = [math.exp(s) for s in scores]
total = sum(exps)
probs = [e / total for e in exps]

print(probs)
print(sum(probs))

[0.9748848013622092, 0.017855637979865575, 0.007259560657925266]
1.0


<a id="s29"></a>

### Cross Entropy: দুই softmax-এর loss

বেশি confident হলে loss কম।

In [11]:
import math

p_a = 0.659   # score [2.0, 1.0, 0.1] থেকে প্রথমটার probability
p_b = 0.975   # score [5.0, 1.0, 0.1] থেকে প্রথমটার probability

loss_a = -math.log(p_a)
loss_b = -math.log(p_b)

print(loss_a)
print(loss_b)

0.41703174447962976
0.025317807984289897


<a id="s30"></a>

### Cross Entropy: p বদলালে loss

In [12]:
import math

for p in [0.9, 0.5, 0.1, 0.01]:
    print(p, -math.log(p))

0.9 0.10536051565782628
0.5 0.6931471805599453
0.1 2.3025850929940455
0.01 4.605170185988091


<a id="s31"></a>

### Streaming: stream_text() with yield

`stream=True` দিয়ে টুকরো টুকরো উত্তর `yield` করে।

In [13]:
def stream_text(prompt):
    stream=client.chat.completions.create(
    model="openai/gpt-oss-20b",
    messages=[{"role": "user", "content": prompt}],
    stream=True,
)

    for chunk in stream:
        piece = chunk.choices[0].delta.content
        if piece:
            yield piece

for part in stream_text("বাংলাদেশ নিয়ে তিন লাইন লেখো"):    
    print(part, end="", flush=True)

বাংলাদেশ দক্ষিণ এশিয়ার একটি সমৃদ্ধ ঐতিহাসিক ও সাংস্কৃতিক দেশ।  
এখানে ঘন বাঁশের বন, গমের ক্ষেত এবং বৈচিত্র্যময় সামুদ্রিক জীবজন্তু দেখা যায়।  
দেশের রাজধানী ঢাকা বিশ্বখ্যাত অর্থনৈতিক ও শিক্ষামূলক কেন্দ্র।

<a id="s32"></a>

### Generator: count_to()

In [14]:
def count_to(n):
    for i in range(1, n + 1):
        yield i

for number in count_to(3):
    print(number)

1
2
3


<a id="s33"></a>

### Generator: yield-এর মাঝে থামা

প্রতিবার `yield`-এ থেমে যায়, `for` চাইলে আবার চলে।

In [15]:
def with_yield():
    print("১ বানাচ্ছি")
    yield 1
    print("২ বানাচ্ছি")
    yield 2
    print("৩ বানাচ্ছি")
    yield 3

for n in with_yield():
    print("পেলাম:", n)

১ বানাচ্ছি
পেলাম: 1
২ বানাচ্ছি
পেলাম: 2
৩ বানাচ্ছি
পেলাম: 3


<a id="s34"></a>

### SSE: to_sse()

প্রতিটা টুকরো `data: ...\n\n` format-এ।

In [16]:
def to_sse(piece):
    return f"data: {piece}\n\n"

for part in stream_text("এক লাইনে আসসালামু বলো"):
    print(repr(to_sse(part)))

'data: আ\n\n'
'data: স\n\n'
'data: স\n\n'
'data: াল\n\n'
'data: াম\n\n'
'data: ু\n\n'
'data:  আল\n\n'
'data: াইক\n\n'
'data: ুম\n\n'
'data: !\n\n'


<a id="s35"></a>

### ask(): history ছাড়া

দুই প্রশ্ন আলাদা পাঠালে model নাম মনে রাখে না।

In [17]:
def ask(messages):
    r = client.chat.completions.create(
        model="openai/gpt-oss-20b",
        messages=messages,
    )
    return r.choices[0].message.content

print(ask([{"role": "user", "content": "আমার নাম রফিক।"}]))
print(ask([{"role": "user", "content": "আমার নাম কী?"}]))

হ্যালো রফিক! আপনাকে স্বাগতম। কীভাবে আমি আপনাকে সাহায্য করতে পারি?
দুঃখিত, আমার কাছে আপনার নাম নেই। অনুগ্রহ করে জানালে আমি আপনাকে সেই নাম দিয়ে সম্বোধন করতে পারি।


<a id="s36"></a>

### chat(): history সহ

`history` list-এ সব কথা জমে, তাই এবার নাম মনে থাকে।

In [18]:
history = []

def chat(user_text):
    history.append({"role": "user", "content": user_text})
    reply = ask(history)
    history.append({"role": "assistant", "content": reply})
    return reply

print(chat("আমার নাম রফিক।"))
print(chat("আমার নাম কী?"))

হ্যালো রফিক! আপনাকে পেয়ে ভালো লাগল। আজ আপনি কী নিয়ে আলোচনা করতে চান বা কী ধরনের সহায়তা প্রয়োজন? আপনার প্রশ্ন বা বিষয় জানালে আমি সর্বোত্তমভাবে সাহায্য করতে প্রস্তুত।
আপনার নাম রফিক।


<a id="s37"></a>

### history-র দৈর্ঘ্য

In [19]:
print(len(history))

4


<a id="s38"></a>

### history-র ভেতরে কী আছে

In [20]:
for m in history:
    print(m["role"], "->", m["content"][:30])

user -> আমার নাম রফিক।
assistant -> হ্যালো রফিক! আপনাকে পেয়ে ভালো
user -> আমার নাম কী?
assistant -> আপনার নাম রফিক।


<a id="s39"></a>

### history[-N:]: শেষের N টা message

In [21]:
MAX_MESSAGES = 10

def chat(user_text):
    history.append({"role": "user", "content": user_text})
    recent = history[-MAX_MESSAGES:]          # শুধু শেষের ৬টা
    reply = ask(recent)
    history.append({"role": "assistant", "content": reply})
    return reply

<a id="s40"></a>

### পরীক্ষা: নাম মনে থাকে কি?

`MAX_MESSAGES` ছোট করলে কী হয় দেখার জায়গা।

In [22]:
history = []
chat("আমার নাম রফিক।")
chat("আমি ঢাকায় থাকি।")
chat("আমার পেশা শিক্ষক।")
chat("আমার প্রিয় রং নীল।")
print(chat("আমার নাম কী?"))

আপনার নাম রফিক!


<a id="s41"></a>

### save_history() আর load_history()

`ensure_ascii=False` দিয়ে বাংলা পড়ার মতো রাখা হয়েছে।

In [23]:
import json

def save_history(history, path="chat.json"):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(history, f, ensure_ascii=False, indent=2)

def load_history(path="chat.json"):
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)

save_history(history)
print(load_history())

[{'role': 'user', 'content': 'আমার নাম রফিক।'}, {'role': 'assistant', 'content': 'হ্যালো রফিক! 😊 কীভাবে সাহায্য করতে পারি?'}, {'role': 'user', 'content': 'আমি ঢাকায় থাকি।'}, {'role': 'assistant', 'content': 'দারুণ! ঢাকা শহরে থাকাকালীন আপনি কী বিষয়ে সহায়তা চান? কোনো তথ্য, পরামর্শ, বা শুধু আলাপ করতে চান? আমাকে জানান, আমি সাহায্য করতে প্রস্তুত আছি।'}, {'role': 'user', 'content': 'আমার পেশা শিক্ষক।'}, {'role': 'assistant', 'content': 'শুনে ভালো লাগছে! আপনি কোন বিষয় বা শ্রেণির শিক্ষক? আপনার পেশা সম্পর্কিত কোনো প্রশ্ন, শিক্ষণ কৌশল, বা রিসোর্স দরকার হলে নির্দ্বিধায় জানাতে পারেন। আপনার কাজকে কীভাবে আরও কার্যকর ও মজাদার করা যায়, সে বিষয়ে আমি আপনাকে কিছু আইডিয়া দিতে পারি। আপনার আগ্রহের দিকটি শেয়ার করুন, চলুন আলাপ শুরু করি!'}, {'role': 'user', 'content': 'আমার প্রিয় রং নীল।'}, {'role': 'assistant', 'content': 'নিশ্চয়ই, নীল একটি চমৎকার রঙ! আকাশ ও সমুদ্রের স্মৃতি এনে দেয়, তাই না? নীলের কোনো বিশেষ ছায়া আপনার পছন্দ, যেমন গাঢ় নীল, ফিরোজা, নাকি হালকা আকাশী? আপনি কি নীল রঙের কিছু জিনিস বা পোশাক 

<a id="s42"></a>

### History আবার load করা

In [24]:
history = []

history = load_history()

print(len(history))

10


<a id="s43"></a>

### sessions: user-ভেদে আলাদা history

In [25]:
sessions = {}

def chat(session_id, user_text):
    history = sessions.setdefault(session_id, [])
    history.append({"role": "user", "content": user_text})
    reply = ask(history[-10:])
    history.append({"role": "assistant", "content": reply})
    return reply

print(chat("rafiq", "আমার নাম রফিক।"))
print(chat("karim", "আমার নাম কী?"))

হ্যালো রফিক! 😊 জানতে পেরে ভালো লাগলো। আপনি আজ কী বিষয়ে সাহায্য চান? আমাকে জানিয়ে দিন, আমি সাহায্য করতে প্রস্তুত।
দুঃখিত, আমার কাছে আপনার নাম নেই। আপনি কি আমাকে বলতে চান?


<a id="s44"></a>

### safe_chat(): প্রথম version

শুধু `try / except`।

In [26]:
def safe_chat(session_id, user_text):
    try:
        return chat(session_id, user_text)
    except Exception as e:
        print("সমস্যা:", e)
        return "দুঃখিত, এখন উত্তর দিতে পারছি না। একটু পরে আবার চেষ্টা করুন।"

print(safe_chat("rafiq", "হ্যালো"))

হ্যালো রফিক! 👋 আজকের দিনটা কেমন কাটছে আপনার? কিছু প্রশ্ন বা কাজ আছে কি, যেটাতে আমি সহায়তা করতে পারি?


<a id="s45"></a>

### safe_chat(): retry সহ

401-এ retry নেই, 429-এ অপেক্ষা করে retry।

In [27]:
import time
from openai import RateLimitError, AuthenticationError

def safe_chat(session_id, user_text, retries=2):
    for attempt in range(retries + 1):
        try:
            return chat(session_id, user_text)
        except AuthenticationError:
            return "সার্ভারের key-তে সমস্যা আছে।"
        except RateLimitError:
            time.sleep(2)
        except Exception as e:
            print("সমস্যা:", e)
            break
    return "দুঃখিত, এখন উত্তর দিতে পারছি না। একটু পরে আবার চেষ্টা করুন।"

print(safe_chat("rafiq", "হ্যালো"))

হ্যালো রফিক! 👋 আজও আপনাকে কীভাবে সহায়তা করতে পারি?


<a id="p4"></a>
<a id="s46"></a>

## ৪. আমার নোট

নিজে লিখে ভরো।

### Gradient Descent মানে কী


### Streaming কেন লাগে


### আজকের সবচেয়ে বড় ভুল